# ChurnGuard — Deployment (Gradio on Google Colab)

This notebook loads the model trained in `churnguardv2.ipynb` (`best_churn_model.pkl`) and wraps it in a **simple web form**: enter a handful of the most important customer details and get back a Churn / Stay prediction with a probability.

**How to use:**
1. Run the cells in order.
2. When prompted (the `files.upload()` cell), upload your `best_churn_model.pkl` file.
3. The last cell gives you a public URL you can open from any device to try the model.

> Note: the form only asks for the ~10 features that matter most (per the EDA). Any field you leave blank is passed to the model as missing — the model (trained with native missing-value handling) treats it exactly as it did during training, no fake defaults invented.


## 1) Install libraries


In [ ]:
!pip install -q gradio catboost joblib pandas numpy scikit-learn

## 2) Upload the trained model (best_churn_model.pkl)

Select the file you saved from the training notebook.


In [ ]:
from google.colab import files
import joblib

uploaded = files.upload()  # choose best_churn_model.pkl
model_filename = list(uploaded.keys())[0]
model = joblib.load(model_filename)
print(f"Model loaded: {model_filename}")
print(model)


## 3) Define the data columns (must match exactly what the model was trained on)

> Note: the model was trained on **raw** features + **engineered** features (`total_overage_charge`, `device_generation_gap`, `revenue_utilization_index`, ...). We don't add these as form fields; instead, `engineer_features()` computes them automatically from whatever the user enters (any engineered feature that needs a raw value the user didn't provide is simply left as `NaN`, same as during training).


In [ ]:
import pandas as pd
import numpy as np

# Exact raw numeric/categorical columns the model was trained on
# (same names/order as the merged Record.csv + Client.csv dataset)
RAW_NUM_COLS = ['rev_Mean', 'mou_Mean', 'totmrc_Mean', 'da_Mean', 'ovrmou_Mean', 'ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean', 'roam_Mean', 'change_mou', 'change_rev', 'drop_vce_Mean', 'drop_dat_Mean', 'blck_vce_Mean', 'blck_dat_Mean', 'unan_vce_Mean', 'unan_dat_Mean', 'plcd_vce_Mean', 'plcd_dat_Mean', 'recv_vce_Mean', 'recv_sms_Mean', 'comp_vce_Mean', 'comp_dat_Mean', 'custcare_Mean', 'ccrndmou_Mean', 'cc_mou_Mean', 'inonemin_Mean', 'threeway_Mean', 'mou_cvce_Mean', 'mou_cdat_Mean', 'mou_rvce_Mean', 'owylis_vce_Mean', 'mouowylisv_Mean', 'iwylis_vce_Mean', 'mouiwylisv_Mean', 'peak_vce_Mean', 'peak_dat_Mean', 'mou_peav_Mean', 'mou_pead_Mean', 'opk_vce_Mean', 'opk_dat_Mean', 'mou_opkv_Mean', 'mou_opkd_Mean', 'drop_blk_Mean', 'attempt_Mean', 'complete_Mean', 'callfwdv_Mean', 'callwait_Mean', 'months', 'uniqsubs', 'actvsubs', 'totcalls', 'totmou', 'totrev', 'adjrev', 'adjmou', 'adjqty', 'avgrev', 'avgmou', 'avgqty', 'avg3mou', 'avg3qty', 'avg3rev', 'avg6mou', 'avg6qty', 'avg6rev', 'hnd_price', 'phones', 'models', 'truck', 'rv', 'lor', 'adults', 'income', 'numbcars', 'forgntvl', 'eqpdays']

CAT_COLS = ['new_cell', 'crclscod', 'asl_flag', 'prizm_social_one', 'area', 'dualband', 'refurb_new', 'hnd_webcap', 'ownrent', 'dwlltype', 'marital', 'infobase', 'HHstatin', 'dwllsize', 'ethnic', 'kid0_2', 'kid3_5', 'kid6_10', 'kid11_15', 'kid16_17', 'creditcd']

# Engineered features, computed exactly as in the training notebook's
# "Feature Engineering" cell — the model needs these alongside the raw ones.
ENGINEERED_NUM_COLS = [
    'total_overage_charge', 'device_generation_gap', 'never_upgraded',
    'revenue_utilization_index', 'silent_drop_indicator', 'call_failure_rate',
    'voice_drop_ratio', 'relative_mou_momentum', 'relative_rev_momentum',
    'overage_intensity', 'phones_per_year', 'handset_depreciation_index',
    'data_usage_ratio', 'peak_usage_ratio', 'inactive_subs_count',
]

NUM_COLS = RAW_NUM_COLS + ENGINEERED_NUM_COLS
ALL_COLS = NUM_COLS + CAT_COLS
print(f"Raw features: {len(RAW_NUM_COLS) + len(CAT_COLS)}")
print(f"Engineered features: {len(ENGINEERED_NUM_COLS)}")
print(f"Total features expected by the model: {len(ALL_COLS)}")


def engineer_features(row: dict) -> dict:
    """
    Computes the exact same engineered features used during training, from
    whatever raw feature values are present in `row` (which may contain NaN
    for any column the user didn't fill in). If a raw value needed for an
    engineered feature is missing, the result is NaN automatically (exactly
    like it would be in the training data for that customer) — no invented
    defaults.
    """
    r = row  # shorthand

    eng = {}
    eng['total_overage_charge'] = r['rev_Mean'] - r['totmrc_Mean']

    device_generation_gap = (r['months'] * 30.4) - r['eqpdays']
    eng['device_generation_gap'] = device_generation_gap
    eng['never_upgraded'] = (
        np.nan if pd.isna(device_generation_gap)
        else (1 if device_generation_gap <= 5 else 0)
    )

    eng['revenue_utilization_index'] = r['mou_Mean'] / (r['rev_Mean'] + 1)
    eng['silent_drop_indicator'] = r['change_mou'] * (1 / (r['custcare_Mean'] + 1))

    eng['call_failure_rate'] = r['drop_blk_Mean'] / (r['attempt_Mean'] + 1)
    eng['voice_drop_ratio'] = r['drop_vce_Mean'] / (r['plcd_vce_Mean'] + 1)

    eng['relative_mou_momentum'] = r['change_mou'] / (r['avgmou'] + 1)
    eng['relative_rev_momentum'] = r['change_rev'] / (r['avgrev'] + 1)

    eng['overage_intensity'] = r['ovrmou_Mean'] / (r['mou_Mean'] + 1)

    eng['phones_per_year'] = r['phones'] / ((r['months'] / 12.0) + 0.1)
    eng['handset_depreciation_index'] = r['hnd_price'] / (r['eqpdays'] + 1)

    total_mou_calculated = r['mou_cvce_Mean'] + r['mou_cdat_Mean']
    eng['data_usage_ratio'] = r['mou_cdat_Mean'] / (total_mou_calculated + 1)

    total_peak_offpeak = r['mou_peav_Mean'] + r['mou_opkv_Mean']
    eng['peak_usage_ratio'] = r['mou_peav_Mean'] / (total_peak_offpeak + 1)

    eng['inactive_subs_count'] = r['uniqsubs'] - r['actvsubs']

    return eng


## 4) Prediction function

Takes only the ~10 values the user fills in on the simplified form, leaves everything else as missing, computes the engineered features, and builds the full row (raw + engineered) in the exact column order/names the model was trained on.


In [ ]:
def predict_churn(months, eqpdays, rev_Mean, mou_Mean, totmrc_Mean,
                   change_mou, custcare_Mean, hnd_price, uniqsubs, actvsubs):

    # Start with an empty row (everything missing) — covers raw, engineered,
    # and categorical columns alike
    row = {c: np.nan for c in ALL_COLS}

    entered = {
        "months": months, "eqpdays": eqpdays, "rev_Mean": rev_Mean, "mou_Mean": mou_Mean,
        "totmrc_Mean": totmrc_Mean, "change_mou": change_mou,
        "custcare_Mean": custcare_Mean, "hnd_price": hnd_price,
        "uniqsubs": uniqsubs, "actvsubs": actvsubs,
    }

    for k, v in entered.items():
        if v is None or v == "":
            continue  # leave it missing
        row[k] = v

    # Compute the engineered features from the same raw values the user
    # entered (or from NaN if a value wasn't given), using the exact same
    # formulas used during training
    row.update(engineer_features(row))

    X_input = pd.DataFrame([row])[ALL_COLS]

    proba = float(model.predict_proba(X_input)[0, 1])
    pred = "\U0001F534 Likely to churn" if proba >= 0.5 else "\U0001F7E2 Likely to stay"

    return f"{pred}\n\nChurn probability: {proba:.1%}"

## 5) Build and launch the Gradio interface


In [ ]:
import gradio as gr

inputs = [
    gr.Number(label="Subscription length (months)", value=12,
               info="How many months this customer has been subscribed"),
    gr.Number(label="Device age (days)", value=365,
               info="Days since the customer's current handset was activated"),
    gr.Number(label="Average monthly revenue ($)", value=50,
               info="Average monthly revenue billed to this customer"),
    gr.Number(label="Average monthly usage (minutes)", value=300,
               info="Average voice minutes used per month"),
    gr.Number(label="Monthly plan charge ($)", value=40,
               info="Fixed recurring monthly charge (before overage)"),
    gr.Number(label="Change in usage last month (minutes)", value=0,
               info="How much monthly usage went up (+) or down (-) recently"),
    gr.Number(label="Customer care calls", value=1,
               info="Number of calls made to customer care"),
    gr.Number(label="Handset price ($)", value=150,
               info="Retail price of the customer's phone"),
    gr.Number(label="Number of lines", value=1,
               info="Total subscriptions/lines on the account"),
    gr.Number(label="Active lines", value=1,
               info="How many of those lines are currently active"),
]

demo = gr.Interface(
    fn=predict_churn,
    inputs=inputs,
    outputs=gr.Textbox(label="Result"),
    title="\U0001F4C9 ChurnGuard — Customer Churn Prediction",
    description="Enter what you know about the customer. Any field you're unsure about can be left at its default — the model handles missing values natively.",
    flagging_mode="never",  # disables the Flag button/mechanism (stops the warnings and dataset*.csv files)
)

demo.launch(share=True, debug=True)
